# 02 — Exploratory Data Analysis & Customer Segmentation

Deep EDA is the centre of this project. Every major visualisation is followed by a **business insight**. Clustering (K-means + hierarchical) is used for segmentation insight, not as the final classifier.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")
print(f"Raw data     : {DATA_RAW}")
print(f"Interim      : {DATA_INTERIM}")
print(f"Processed    : {DATA_PROCESSED}")
print(f"Figures      : {FIGURES_DIR}")
print(f"Models       : {MODELS_DIR}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42
Raw data     : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw
Interim      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim
Processed    : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\processed
Figures      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\reports\figures
Models       : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-custome

In [2]:
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

df = pd.read_csv(DATA_INTERIM / "ecomm_validated.csv")
print(df.shape)
df.head()

(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


## 0. Harmonise known label overlaps (EDA view only)

We create analysis-friendly copies without changing the interim file yet (preprocessing owns the final mapping).

In [3]:
eda = df.copy()

eda["PreferredLoginDevice_H"] = eda["PreferredLoginDevice"].replace({"Phone": "Mobile Phone"})
eda["PreferredPaymentMode_H"] = eda["PreferredPaymentMode"].replace({
    "CC": "Credit Card",
    "COD": "Cash on Delivery",
})
eda["PreferedOrderCat_H"] = eda["PreferedOrderCat"].replace({"Mobile": "Mobile Phone"})

print(eda["PreferredLoginDevice_H"].value_counts())
print(eda["PreferredPaymentMode_H"].value_counts())
print(eda["PreferedOrderCat_H"].value_counts())

PreferredLoginDevice_H
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64
PreferredPaymentMode_H
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64
PreferedOrderCat_H
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: count, dtype: int64


## 1. Missing-value heatmap & outlier scan

In [4]:
miss_cols = eda.columns[eda.isna().any()].tolist()
miss_mat = eda[miss_cols].isna().astype(int)

fig = px.imshow(
    miss_mat.T,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels=dict(x="Row index", y="Feature", color="Missing"),
    title="Missing-value pattern (selected columns)",
)
fig.update_layout(height=350)
fig.write_html(REPORTS_DIR / "02_missing_heatmap.html")
fig.show()

print("Missing % by column:")
print((eda[miss_cols].isna().mean() * 100).round(2))

Missing % by column:
Tenure                         4.69
WarehouseToHome                4.46
HourSpendOnApp                 4.53
OrderAmountHikeFromlastYear    4.71
CouponUsed                     4.55
OrderCount                     4.58
DaySinceLastOrder              5.45
dtype: float64


**Insight:** Missingness is scattered across engagement / order features rather than a single block of rows. That pattern favours **per-column median imputation** over listwise deletion, which would discard too many customers.

In [5]:
num_cols = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp", "NumberOfDeviceRegistered",
    "SatisfactionScore", "NumberOfAddress", "OrderAmountHikeFromlastYear",
    "CouponUsed", "OrderCount", "DaySinceLastOrder", "CashbackAmount",
]

def iqr_outlier_share(s: pd.Series) -> float:
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return float(((s < lo) | (s > hi)).mean())

outlier_tbl = pd.DataFrame({
    "feature": num_cols,
    "outlier_share_iqr": [iqr_outlier_share(eda[c]) for c in num_cols],
    "skew": [eda[c].dropna().skew() for c in num_cols],
}).sort_values("outlier_share_iqr", ascending=False)
outlier_tbl.to_csv(DATA_INTERIM / "02_outlier_summary.csv", index=False)
outlier_tbl

,feature,outlier_share_iqr,skew
8,OrderCount,0.130864,2.196414
7,CouponUsed,0.117045,2.545653
10,CashbackAmount,0.077798,1.149846
3,NumberOfDeviceRegistered,0.070515,-0.396969
9,DaySinceLastOrder,0.011648,1.191000
6,OrderAmountHikeFromlastYear,0.006151,0.790785
2,HourSpendOnApp,0.001116,-0.027213
0,Tenure,0.000745,0.736513
5,NumberOfAddress,0.000710,1.088639
1,WarehouseToHome,0.000372,1.619154


**Insight:** Several spend / distance features are right-skewed with IQR outliers. We keep them for modelling but will use **RobustScaler** so extreme delivery distances or cashback amounts do not dominate linear / neural weights.

## 2. Univariate — target & categoricals

In [6]:
cat_cols = [
    "PreferredLoginDevice_H", "CityTier", "PreferredPaymentMode_H", "Gender",
    "PreferedOrderCat_H", "SatisfactionScore", "MaritalStatus", "Complain",
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
for ax, col in zip(axes, cat_cols):
    vc = eda[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values, color="#457b9d")
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
fig.suptitle("Categorical / ordinal distributions")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_categorical_distributions.png", dpi=150)
plt.show()

**Insight:** Debit/Credit cards dominate payments; Married is the largest marital segment; Laptop & Accessory and Mobile Phone dominate order categories. Rare payment modes (UPI, E-wallet) are still large enough to estimate churn rates separately.

In [7]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.ravel()
for ax, col in zip(axes, num_cols):
    ax.hist(eda[col].dropna(), bins=30, color="#1d3557", alpha=0.85)
    ax.set_title(col, fontsize=9)
for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")
fig.suptitle("Numerical distributions")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_numerical_distributions.png", dpi=150)
plt.show()

**Insight:** Tenure, OrderCount, CouponUsed, and DaySinceLastOrder are heavily right-skewed — typical of retail behavioural data. Median-centred summaries and robust scaling are more appropriate than means / StandardScaler alone.

## 3. Bivariate — churn rate by every feature

In [8]:
def churn_rate_by(col: str) -> pd.DataFrame:
    g = eda.groupby(col, dropna=False)["Churn"].agg(["mean", "count"])
    g = g.rename(columns={"mean": "churn_rate", "count": "n"}).reset_index()
    return g.sort_values("churn_rate", ascending=False)

for col in cat_cols:
    tbl = churn_rate_by(col)
    print("\n===", col, "===")
    print(tbl.to_string(index=False))
    fig = px.bar(
        tbl, x=col, y="churn_rate", text="n",
        title=f"Churn rate by {col}",
        labels={"churn_rate": "Churn rate"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(yaxis_tickformat=".0%")
    fig.write_html(REPORTS_DIR / f"02_churn_by_{col}.html")
    fig.show()


=== PreferredLoginDevice_H ===
PreferredLoginDevice_H  churn_rate    n
              Computer    0.198286 1634
          Mobile Phone    0.156156 3996



=== CityTier ===
 CityTier  churn_rate    n
        3    0.213705 1722
        2    0.198347  242
        1    0.145117 3666



=== PreferredPaymentMode_H ===
PreferredPaymentMode_H  churn_rate    n
      Cash on Delivery    0.249027  514
              E wallet    0.228013  614
                   UPI    0.173913  414
            Debit Card    0.153846 2314
           Credit Card    0.142052 1774



=== Gender ===
Gender  churn_rate    n
  Male    0.177305 3384
Female    0.154942 2246



=== PreferedOrderCat_H ===
PreferedOrderCat_H  churn_rate    n
      Mobile Phone    0.274038 2080
           Fashion    0.154964  826
Laptop & Accessory    0.102439 2050
            Others    0.075758  264
           Grocery    0.048780  410



=== SatisfactionScore ===
 SatisfactionScore  churn_rate    n
                 5    0.238267 1108
                 3    0.171967 1698
                 4    0.171322 1074
                 2    0.126280  586
                 1    0.115120 1164



=== MaritalStatus ===
MaritalStatus  churn_rate    n
       Single    0.267261 1796
     Divorced    0.146226  848
      Married    0.115204 2986



=== Complain ===
 Complain  churn_rate    n
        1    0.316708 1604
        0    0.109290 4026


**Insight (complaints):** Customers with `Complain=1` churn far more often. Complaint resolution is the most actionable operational lever in this dataset.

**Insight (satisfaction):** Lower satisfaction scores show elevated churn — combine with complaints for a high-risk segment.

**Insight (device / payment / city):** Computer login and some payment modes show different churn rates; CityTier differences are smaller but still worth controlling for in models.

In [9]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.ravel()
for ax, col in zip(axes, num_cols):
    data = [eda.loc[eda["Churn"] == 0, col].dropna(), eda.loc[eda["Churn"] == 1, col].dropna()]
    ax.boxplot(data, labels=["Retained", "Churned"], showfliers=False)
    ax.set_title(col, fontsize=9)
for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")
fig.suptitle("Numeric features by churn (boxplots, outliers hidden)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_boxplots_by_churn.png", dpi=150)
plt.show()

**Insight:** Churners tend toward **shorter tenure**, often **more complaints**, and different recency / cashback profiles. Tenure and complaint status should be first-class model features and retention triggers.

## 4. Statistical association tests

In [10]:
def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

cat_assoc = []
for col in cat_cols:
    ct = pd.crosstab(eda[col], eda["Churn"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    cat_assoc.append({"feature": col, "chi2": chi2, "p_value": p, "cramers_v": cramers_v(eda[col], eda["Churn"])})
cat_assoc_df = pd.DataFrame(cat_assoc).sort_values("cramers_v", ascending=False)
cat_assoc_df.to_csv(DATA_INTERIM / "02_categorical_associations.csv", index=False)
cat_assoc_df

,feature,chi2,p_value,cramers_v
7,Complain,350.925455,2.664461e-78,0.249662
4,PreferedOrderCat_H,288.597786,3.119243e-61,0.226408
6,MaritalStatus,188.671040,1.073011e-41,0.183062
5,SatisfactionScore,69.865388,2.423335e-14,0.111398
2,PreferredPaymentMode_H,51.828960,1.497857e-10,0.095947
1,CityTier,40.982404,1.261200e-09,0.085319
0,PreferredLoginDevice_H,14.401253,1.477040e-04,0.050576
3,Gender,4.662908,3.082094e-02,0.028779


**Insight:** Cramer's V ranks which categorical attributes carry the strongest association with churn (expect **Complain** near the top). Statistically significant associations justify keeping those features through GA/PSO selection rather than dropping them early.

In [11]:
anova_rows = []
for col in num_cols:
    a = eda.loc[eda["Churn"] == 0, col].dropna()
    b = eda.loc[eda["Churn"] == 1, col].dropna()
    # Kruskal-Wallis is safer under skew / unequal variance
    h, p = stats.kruskal(a, b)
    anova_rows.append({"feature": col, "kruskal_H": h, "p_value": p})
anova_df = pd.DataFrame(anova_rows).sort_values("kruskal_H", ascending=False)
anova_df.to_csv(DATA_INTERIM / "02_numeric_kruskal.csv", index=False)
anova_df

,feature,kruskal_H,p_value
0,Tenure,878.092864,5.678503e-193
9,DaySinceLastOrder,185.565082,2.954178e-42
10,CashbackAmount,167.564739,2.518104e-38
4,SatisfactionScore,61.826091,3.751701e-15
3,NumberOfDeviceRegistered,57.699428,3.053916e-14
1,WarehouseToHome,35.406192,2.676347e-09
5,NumberOfAddress,4.681090,3.049639e-02
8,OrderCount,4.379986,3.636331e-02
6,OrderAmountHikeFromlastYear,2.294310,1.298489e-01
2,HourSpendOnApp,1.460878,2.267904e-01


In [12]:
# Mutual information (mixed types via label encoding for cats)
mi_df = eda.dropna().copy()
X_mi = mi_df.drop(columns=["CustomerID", "Churn", "PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"])
# use harmonised cats
for c in ["PreferredLoginDevice_H", "PreferredPaymentMode_H", "PreferedOrderCat_H", "Gender", "MaritalStatus"]:
    X_mi[c] = LabelEncoder().fit_transform(X_mi[c].astype(str))
y_mi = mi_df["Churn"].values
mi = mutual_info_classif(X_mi, y_mi, random_state=RANDOM_SEED)
mi_tbl = pd.DataFrame({"feature": X_mi.columns, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
mi_tbl.to_csv(DATA_INTERIM / "02_mutual_information.csv", index=False)

fig = px.bar(mi_tbl, x="mutual_info", y="feature", orientation="h", title="Mutual information with Churn")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.write_html(REPORTS_DIR / "02_mutual_information.html")
fig.show()
mi_tbl.head(10)

,feature,mutual_info
14,CashbackAmount,0.138298
0,Tenure,0.132355
17,PreferedOrderCat_H,0.029771
9,Complain,0.026810
7,MaritalStatus,0.020606
13,DaySinceLastOrder,0.010453
5,NumberOfDeviceRegistered,0.008809
2,WarehouseToHome,0.005083
11,CouponUsed,0.005051
12,OrderCount,0.004013


**Insight:** Mutual information surfaces non-linear dependencies that correlation alone can miss. Features high on MI and Kruskal/Cramer's V are strong candidates to survive GA/PSO feature selection.

## 5. Correlations & interactions

In [13]:
corr = eda[num_cols + ["Churn"]].corr(method="spearman")
fig = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r",
                title="Spearman correlation (numerics + Churn)")
fig.update_layout(height=600)
fig.write_html(REPORTS_DIR / "02_spearman_corr.html")
fig.show()

# Simple interaction: Complain x Satisfaction
inter = eda.groupby(["Complain", "SatisfactionScore"])["Churn"].mean().reset_index()
fig2 = px.density_heatmap(
    eda, x="SatisfactionScore", y="Complain", z="Churn", histfunc="avg",
    title="Average churn: Complain × SatisfactionScore",
    color_continuous_scale="YlOrRd",
)
fig2.write_html(REPORTS_DIR / "02_complain_satisfaction_interaction.html")
fig2.show()

**Insight:** The **Complain × Satisfaction** interaction is a classic retention segment: unhappy complainers deserve priority outreach. We will engineer an explicit interaction flag in preprocessing.

## 6. Behavioural deep-dives

In [14]:
# Tenure cohorts
eda["TenureCohort"] = pd.cut(
    eda["Tenure"], bins=[-0.1, 3, 9, 15, eda["Tenure"].max() + 1],
    labels=["New(0-3)", "Early(3-9)", "Mid(9-15)", "Long(15+)"],
)
cohort = eda.groupby("TenureCohort", observed=False)["Churn"].agg(["mean", "count"]).reset_index()
fig = px.bar(cohort, x="TenureCohort", y="mean", text="count", title="Churn rate by tenure cohort")
fig.update_layout(yaxis_title="Churn rate", yaxis_tickformat=".0%")
fig.write_html(REPORTS_DIR / "02_tenure_cohorts.html")
fig.show()
cohort

,TenureCohort,mean,count
0,New(0-3),0.418590,1560
1,Early(3-9),0.066616,1321
2,Mid(9-15),0.061538,1105
3,Long(15+),0.042029,1380


**Insight:** New customers (0–3 months) are the riskiest tenure band. Onboarding quality, early coupons, and first-delivery experience matter more than late-stage loyalty campaigns for this segment.

In [15]:
# Cashback / coupons / orders / recency / warehouse
for col, title in [
    ("CashbackAmount", "Cashback vs churn"),
    ("CouponUsed", "Coupons vs churn"),
    ("OrderCount", "Order count vs churn"),
    ("DaySinceLastOrder", "Days since last order vs churn"),
    ("WarehouseToHome", "Warehouse distance vs churn"),
]:
    fig = px.histogram(eda, x=col, color=eda["Churn"].map({0: "Retained", 1: "Churned"}),
                       barmode="overlay", nbins=40, opacity=0.65, title=title)
    fig.write_html(REPORTS_DIR / f"02_{col}_by_churn.html")
    fig.show()

**Insight:** Recency (`DaySinceLastOrder`) and reward intensity (`CashbackAmount`) separate dormant-at-risk customers from active rewarded buyers. Distance (`WarehouseToHome`) links logistics experience to retention — useful for ops, not only marketing.

## 7. Customer segmentation (K-means + hierarchical)

In [16]:
seg_features = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp", "NumberOfDeviceRegistered",
    "SatisfactionScore", "Complain", "OrderCount", "DaySinceLastOrder",
    "CouponUsed", "CashbackAmount",
]
seg = eda[seg_features + ["Churn", "CustomerID"]].dropna().copy()
X_seg = StandardScaler().fit_transform(seg[seg_features])

inertias, sils = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_seg)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_seg, labels))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(list(K_range), inertias, marker="o")
ax[0].set_title("Elbow (inertia)")
ax[0].set_xlabel("k")
ax[1].plot(list(K_range), sils, marker="o", color="#e76f51")
ax[1].set_title("Silhouette")
ax[1].set_xlabel("k")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_kmeans_elbow_silhouette.png", dpi=150)
plt.show()
print(list(zip(K_range, np.round(sils, 3))))

[(2, np.float64(0.257)), (3, np.float64(0.154)), (4, np.float64(0.129)), (5, np.float64(0.144)), (6, np.float64(0.135)), (7, np.float64(0.13))]


In [17]:
best_k = int(list(K_range)[int(np.argmax(sils))])
print("Selected k by silhouette:", best_k)
km = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init=10)
seg["cluster"] = km.fit_predict(X_seg)

profile = seg.groupby("cluster")[seg_features + ["Churn"]].mean()
profile["n"] = seg.groupby("cluster").size()
profile.to_csv(DATA_INTERIM / "02_kmeans_profiles.csv")
print(profile)

# Hierarchical (sample for dendrogram readability)
sample = X_seg[np.random.RandomState(RANDOM_SEED).choice(len(X_seg), size=min(400, len(X_seg)), replace=False)]
Z = linkage(sample, method="ward")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, truncate_mode="level", p=4, ax=ax, no_labels=True)
ax.set_title("Hierarchical clustering dendrogram (Ward, sample)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_hierarchical_dendrogram.png", dpi=150)
plt.show()

# Compare agglomerative labels vs kmeans churn rates
agg = AgglomerativeClustering(n_clusters=best_k)
seg["hcluster"] = agg.fit_predict(X_seg)
print("\nK-means churn by cluster:")
print(seg.groupby("cluster")["Churn"].mean())
print("Hierarchical churn by cluster:")
print(seg.groupby("hcluster")["Churn"].mean())

Selected k by silhouette: 2
            Tenure  WarehouseToHome  HourSpendOnApp  NumberOfDeviceRegistered  \
cluster                                                                         
0         8.611486        15.740786        2.966523                  3.747236   
1        13.426564        15.185185        3.080460                  3.849298   

         SatisfactionScore  Complain  OrderCount  DaySinceLastOrder  \
cluster                                                               
0                 3.042383  0.282555    1.951167           3.768120   
1                 3.121328  0.272031    7.446999           8.606641   

         CouponUsed  CashbackAmount     Churn     n  
cluster                                              
0          1.210995      164.872171  0.170147  3256  
1          4.286079      209.229119  0.116220   783  



K-means churn by cluster:
cluster
0    0.170147
1    0.116220
Name: Churn, dtype: float64
Hierarchical churn by cluster:
hcluster
0    0.178388
1    0.111210
Name: Churn, dtype: float64


**Insight:** Clusters with short tenure + complaints + low satisfaction typically show the highest churn rate. Treat these as **priority retention segments** for CRM campaigns. Hierarchical structure confirms that a small number of behavioural archetypes is enough — we do not need dozens of micro-segments for actionability.

## 8. Optional profiling reports

In [18]:
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(eda.drop(columns=["TenureCohort"], errors="ignore"), title="E-commerce Churn Profiling", minimal=True)
    profile.to_file(REPORTS_DIR / "02_ydata_profile.html")
    print("Wrote ydata profile")
except Exception as e:
    print("ydata-profiling skipped:", e)

try:
    import sweetviz as sv
    report = sv.analyze(eda.drop(columns=["TenureCohort"], errors="ignore"), target_feat="Churn")
    report.show_html(str(REPORTS_DIR / "02_sweetviz.html"), open_browser=False)
    print("Wrote sweetviz report")
except Exception as e:
    print("sweetviz skipped:", e)

ydata-profiling skipped: No module named 'ydata_profiling'
sweetviz skipped: No module named 'sweetviz'


## 9. EDA findings → modelling hypotheses

1. **Complain** and **low SatisfactionScore** jointly identify urgent churn risk → keep both + interaction feature.
2. **Short Tenure** cohorts churn more → tenure bins / nonlinear models (RF, TabNet) should help.
3. Label overlaps in device / payment / category will inflate dimensionality if not harmonised → clean in preprocessing.
4. Skewed numerics + outliers → RobustScaler; avoid dropping outliers that may be real VIP / remote customers.
5. Missing engagement fields are MAR-like → median/mode imputation is sufficient for baselines and TabNet.
6. Clusters give **business segments**; supervised models still own prediction. GA/PSO should be allowed to drop weak demographics if CV F1 improves.

**Next:** Notebook `03` — preprocessing pipeline and train/test split.